In [27]:
%%capture

import torch
from torch.nn.functional import cross_entropy
from transformers import DataCollatorForTokenClassification
import import_ipynb
from a_glance_at_dataset_and_tokenizer import xlmr_tokenizer
from create_model import device, XLMRobertaForTokenClassification
from tokenizing_text import panx_de_encoded

In [28]:
xlmr_model_name = "xlm-roberta-base"
model_name = f"{xlmr_model_name}-finetuned-panx-de"

In [29]:
data_collator = DataCollatorForTokenClassification(xlmr_tokenizer)
trainer = XLMRobertaForTokenClassification.from_pretrained(
    model_name
).to(device)

In [30]:
def forward_pass_with_label(batch):
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]
    batch = data_collator(features)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    with torch.no_grad():
        output = trainer.model(input_ids, attention_mask)
        predicted_label = torch.argmax(output.logits, dim=-1).cpu().numpy()
    loss = cross_entropy(
        output.logits.view(-1, 7), labels.view(-1), reduction="none"
    )
    loss = loss.view(len(input_ids), -1).cpu().numpy()

    return {"loss": loss, "predicted_label": predicted_label}

In [31]:
valid_set = panx_de_encoded["validation"]
valid_set = valid_set.map(forward_pass_with_label, batched=True, batch_size=32)
df = valid_set.to_pandas()

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

AttributeError: 'XLMRobertaForTokenClassification' object has no attribute 'model'